# Runbook de comandos — setup + validação do monorepo

Este notebook centraliza os comandos usados na sessão para corrigir ambiente Python/venv e validar o monorepo.

Use as células em ordem. As etapas que exigem root estão destacadas.

## 1) Helpers de execução

In [1]:
from __future__ import annotations

import os
import shlex
import subprocess
from pathlib import Path

ROOT = Path.cwd().resolve()
print('PWD:', ROOT)

def run(cmd: str, *, check: bool = True) -> int:
    print(f'\n$ {cmd}')
    p = subprocess.run(cmd, shell=True, cwd=ROOT)
    if check and p.returncode != 0:
        raise RuntimeError(f'Falhou com exit code {p.returncode}: {cmd}')
    return p.returncode

PWD: /home/daniel/monorepo-ai-llm/spec


## 2) Diagnóstico rápido do ambiente

In [ ]:
run('id -u && whoami')
run('command -v python3 && python3 --version')
run('pwd')

## 3) Preparar suporte a venv no WSL (requer root)

Se sua sessão já estiver como root, rode direto.
Se não estiver, abra root no terminal antes (`sudo -i`).

In [ ]:
run('apt-get update && apt-get install -y python3.12-venv')

## 4) Recriar `.venv1` e instalar kernel Jupyter

In [ ]:
run('rm -rf .venv1')
run('python3 -m venv .venv1')
run('.venv1/bin/python -m pip install -U pip setuptools wheel ipykernel')
run('.venv1/bin/python -m pip --version')
run('.venv1/bin/python -m pip show ipykernel')

## 5) Validar arquivos-chave do monorepo

In [ ]:
for rel in [
    'package.json',
    'nest-cli.json',
    'apps',
    'packages',
    'spec/notebooks_02_validate_current_config_state (1).ipynb',
]:
    p = ROOT / rel
    print(('✅' if p.exists() else '❌'), rel)

## 6) Executar notebook de validação por linha de comando (opcional)

Gera um arquivo executado com saída em `spec/out/` sem sobrescrever o original.

In [ ]:
run('mkdir -p spec/out')
run('.venv1/bin/python -m pip install -U jupyter nbconvert')
run(".venv1/bin/jupyter nbconvert --to notebook --execute --inplace 'spec/notebooks_02_validate_current_config_state (1).ipynb'", check=False)

## 7) Comandos usados nesta sessão (referência)

- `id -u && whoami`
- `python3 --version`
- `apt-get update && apt-get install -y python3.12-venv`
- `rm -rf .venv1`
- `python3 -m venv .venv1`
- `.venv1/bin/python -m pip install -U pip setuptools wheel ipykernel`
- `.venv1/bin/python -m pip --version`
- `.venv1/bin/python -m pip show ipykernel`
- execução do notebook de validação em VS Code/Jupyter